In [1]:
import os
import math
import time
import torch
import pickle
import numpy as np

import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import torch.autograd as autograd

import torch.nn.functional as functional
import torch.optim.lr_scheduler as lr_scheduler
from utils import *

os.makedirs('./results/many_well_64', exist_ok=True)
torch.set_default_dtype(torch.float32) 
_ = torch.manual_seed(0)

TORCH DEVICE: cuda


In [2]:
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [3]:
if torch.cuda.is_available():
    torch_device = 'cuda'
else:
    torch_device = 'cpu'
print(f"TORCH DEVICE: {torch_device}")

TORCH DEVICE: cuda


In [4]:
class MLPScore(nn.Module):
    def __init__(self):
        super().__init__()
        self.main = nn.Sequential(
            nn.Linear(64, 1024),
            nn.Tanh(),
            nn.Linear(1024,1024),
            nn.Tanh(),
            nn.Linear(1024,1024),
            nn.Tanh(),
            nn.Linear(1024,1024),
            nn.Tanh(),
            nn.Linear(1024,64)
        )

    def forward(self, x):
        score = self.main(x)

        return score

In [5]:
def jacobian(f, x):
    """
    Computes the Jacobian
    """
    B, N = x.shape
    y = f(x)
    jacobian = list()
    for i in range(N):
        v = torch.zeros_like(y)
        v[:, i] = 1.
        dy_i_dx = autograd.grad(y, x, grad_outputs=v, retain_graph=True,
                                create_graph=True, allow_unused=True)[0]  # shape [B, N]
        jacobian.append(dy_i_dx)
    jacobian = torch.stack(jacobian, dim=2).requires_grad_()
    return jacobian

In [6]:
def score_matching(model, samples):
    samples.requires_grad_(True)
    logp = model(samples)
    norm_jacobian = torch.norm(logp, dim=-1) ** 2 / 2.
    jacob_mat = jacobian(model, samples)
    #print(jacob_mat.shape)
    trace_jacobian = torch.diagonal(jacob_mat, dim1=-2, dim2=-1).sum(-1)
    loss = (trace_jacobian + norm_jacobian).mean(-1)
    return loss

In [7]:
dim = 64
filename = "many_well_dataset_64_110000_samples.pkl"
f = open(filename, 'rb')
dataset = pickle.load(f,encoding="latin1")
test_filename = "many_well_dataset_64_20000_test_samples.pkl"
t = open(test_filename,'rb')
testset = pickle.load(t,encoding="latin1")

In [8]:
dataset.shape

torch.Size([110000, 64])

In [9]:
testset.shape

torch.Size([20000, 64])

In [10]:
batch_size = 512
no_training_samples = 100000
no_validation_samples = 10000
no_test_samples = 20000


train_dataloader = torch.utils.data.DataLoader(dataset, batch_size = batch_size, shuffle=True)
val_dataloader = torch.utils.data.DataLoader(dataset[100000:], batch_size = batch_size, shuffle=False)

test_dataloader = torch.utils.data.DataLoader(testset, batch_size = batch_size, shuffle=False)


In [11]:
model = MLPScore().to(torch_device)

In [12]:
PATH = "./results/many_well_64/score_estimation_model_100000.pt"

In [37]:
#For resuming the checkpoint:
checkpoint = torch.load(PATH)
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

In [30]:
optimizer = optim.Adam(model.parameters(),lr= 1.e-5)
#scheduler = lr_scheduler.StepLR(optimizer,step_size = 10, gamma = 0.9)


In [34]:
epochs =  100

for i in range(epochs):
    model.train()
    train_score_loss = 0.
    val_score_loss = 0.
    
    for x in train_dataloader:
        loss = score_matching(model, x)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_score_loss += loss.item()*(x.shape[0])
        
  
    # model.eval()
    # with torch.no_grad():
    for x in val_dataloader:
        val_loss = score_matching(model, x)
        val_score_loss += val_loss.item()*(x.shape[0])
            
    avg_train_loss = train_score_loss/no_training_samples
    avg_val_loss   = val_score_loss/no_validation_samples
            
    template = 'Epoch {:3d}, Train loss: {:.4f},Val loss: {:.4f}'
    print (template.format(i+1 , avg_train_loss , avg_val_loss))
    


Epoch   1, Train loss: -459.4628,Val loss: -419.7145
Epoch   2, Train loss: -459.6379,Val loss: -419.6945
Epoch   3, Train loss: -459.7714,Val loss: -419.9511
Epoch   4, Train loss: -459.9115,Val loss: -420.2716
Epoch   5, Train loss: -460.0251,Val loss: -420.2822
Epoch   6, Train loss: -460.1555,Val loss: -420.2625
Epoch   7, Train loss: -460.3231,Val loss: -420.5029
Epoch   8, Train loss: -460.4373,Val loss: -420.5025
Epoch   9, Train loss: -460.5756,Val loss: -420.5195
Epoch  10, Train loss: -460.6904,Val loss: -420.9146
Epoch  11, Train loss: -460.8864,Val loss: -420.9520
Epoch  12, Train loss: -461.0190,Val loss: -420.8478
Epoch  13, Train loss: -461.1814,Val loss: -421.4465
Epoch  14, Train loss: -461.3000,Val loss: -421.4989
Epoch  15, Train loss: -461.4427,Val loss: -421.6645
Epoch  16, Train loss: -461.5952,Val loss: -421.6946
Epoch  17, Train loss: -461.6589,Val loss: -421.7304
Epoch  18, Train loss: -461.8435,Val loss: -421.9387
Epoch  19, Train loss: -461.9608,Val loss: -42

In [38]:
no_test_samples  = 20000
test_score_loss = 0.
for x in test_dataloader:
    test_loss = score_matching(model, x)
    test_score_loss += test_loss.item()*(x.shape[0])
    
avg_test_loss   = test_score_loss/no_test_samples
print(f"Test Score Loss:{avg_test_loss:.4f}")

Test Score Loss:-341.8284


In [33]:
torch.save({'model_state_dict': model.state_dict(),'optimizer_state_dict': optimizer.state_dict(),}, PATH)

In [ ]:
model(x[:2])

In [ ]:
def grad_dist(x):
    x = x.cpu().detach()
    score = []
    for i in range(x.size()[1]//2):
        y1 = - 4*x[:,2*i]**3 + 12*x[:,2*i] + 0.5
        y2 = - x[:,2*i+1]
        score.append(y1)
        score.append(y2)
    return torch.tensor(np.array(score).T)

In [ ]:
x[:2]

In [ ]:
grad_dist(x[:2])

In [39]:
filename = "many_well_data_64 dim_trained_with_10000_samples.pkl"
f = open(filename, 'rb')
dataset = pickle.load(f,encoding="latin1")

In [40]:
dataset[0].shape

(20000, 64)

In [41]:
data_many_well_64 = torch.tensor(dataset[0] , device = torch_device)

In [ ]:
# neg_grad = -grad_dist(data_many_well_64)

In [ ]:
#neg_grad.shape

In [ ]:
# array = [data_many_well_64.cpu().detach().numpy(), neg_grad.numpy()]

In [42]:
neg_grad_model = - model(data_many_well_64).cpu().detach().numpy()

In [43]:
array = [data_many_well_64.cpu().detach().numpy(), neg_grad_model]

In [44]:
output = open("many_well_data_64_dim_with_neg_score_through_score_model_100000.pkl", 'wb')
pickle.dump(array, output)
output.close()

In [ ]:
#many_well_data_2 dim_with_neg_score_through_log_pdf.pkl
#many_well_data_32 dim_with_neg_score_through_score_model_10000.pkl